In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

# Create a simple payments table to simulate Bronze layer
# In production this would read from Kafka directly 
# Community Edition doesn't support external Kafka connections
# so we simulate the data here 

spark = SparkSession.builder.appName("BronzeIngestion").getOrCreate()

raw_payments = [
    ("pay_a3f9b2c1", "usr_7829kdf9", "mer_4421abc", 149.99, "USD", "credit_card", "succeeded", "2024-01-15T14:32:11", "mobile_ios"),
    ("pay_b7d2e4f1", "usr_3312xyz", "mer_8821def", 23.50, "USD", "debit_card", "failed", "2024-01-15T14:32:12", "web_checkout"),
    ("pay_c9f1a3b2", "usr_9912abc", "mer_1121ghi", 567.00, "EUR", "bank_transfer", "succeeded", "2024-01-15T14:32:13", "pos_terminal"),
    ("pay_d2e4f6a1", "usr_4423def", "mer_3321jkl", 89.99, "GBP", "digital_wallet", "pending", "2024-01-15T14:32:14", "mobile_android"),
    ("pay_e5f7a9b3", "usr_7756ghi", "mer_5521mno", 234.00, "USD", "credit_card", "succeeded", "2024-01-15T14:32:15", "web_checkout"),
    ("pay_a3f9b2c1", "usr_7829kdf9", "mer_4421abc", 149.99, "USD", "credit_card", "succeeded", "2024-01-15T14:32:11", "mobile_ios"),
]

columns = [
    "payment_id", "user_id", "merchant_id", "amount",
    "currency", "payment_method", "status",
    "timestamp", "source_system"
]

df = spark.createDataFrame(raw_payments, columns)
df = df.withColumn("ingest_timestamp", current_timestamp())
df = df.withColumn("layer", lit("bronze"))

print("Bronze layer - raw payment events:")
df.show(truncate=False)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_payments")

print("Bronze table saved successfully")
print(f"Total records ingested: {df.count()}")

Bronze layer - raw payment events:
+------------+------------+-----------+------+--------+--------------+---------+-------------------+--------------+--------------------------+------+
|payment_id  |user_id     |merchant_id|amount|currency|payment_method|status   |timestamp          |source_system |ingest_timestamp          |layer |
+------------+------------+-----------+------+--------+--------------+---------+-------------------+--------------+--------------------------+------+
|pay_a3f9b2c1|usr_7829kdf9|mer_4421abc|149.99|USD     |credit_card   |succeeded|2024-01-15T14:32:11|mobile_ios    |2026-04-12 19:27:06.864277|bronze|
|pay_b7d2e4f1|usr_3312xyz |mer_8821def|23.5  |USD     |debit_card    |failed   |2024-01-15T14:32:12|web_checkout  |2026-04-12 19:27:06.864277|bronze|
|pay_c9f1a3b2|usr_9912abc |mer_1121ghi|567.0 |EUR     |bank_transfer |succeeded|2024-01-15T14:32:13|pos_terminal  |2026-04-12 19:27:06.864277|bronze|
|pay_d2e4f6a1|usr_4423def |mer_3321jkl|89.99 |GBP     |digital_wa

In [0]:
# Verify Bronze table was created correctly
bronze_df = spark.read.format("delta").table("bronze_payments")

print("Total rows in Bronze (including duplicates):")
print(bronze_df.count())

print("\nAll records:")
bronze_df.show(truncate=False)

print("\nChecking for duplicates on payment_id:")
from pyspark.sql.functions import count
bronze_df.groupBy("payment_id") \
    .agg(count("*").alias("count")) \
    .filter("count > 1") \
    .show()

Total rows in Bronze (including duplicates):
6

All records:
+------------+------------+-----------+------+--------+--------------+---------+-------------------+--------------+--------------------------+------+
|payment_id  |user_id     |merchant_id|amount|currency|payment_method|status   |timestamp          |source_system |ingest_timestamp          |layer |
+------------+------------+-----------+------+--------+--------------+---------+-------------------+--------------+--------------------------+------+
|pay_a3f9b2c1|usr_7829kdf9|mer_4421abc|149.99|USD     |credit_card   |succeeded|2024-01-15T14:32:11|mobile_ios    |2026-04-12 19:27:26.127284|bronze|
|pay_b7d2e4f1|usr_3312xyz |mer_8821def|23.5  |USD     |debit_card    |failed   |2024-01-15T14:32:12|web_checkout  |2026-04-12 19:27:26.127284|bronze|
|pay_c9f1a3b2|usr_9912abc |mer_1121ghi|567.0 |EUR     |bank_transfer |succeeded|2024-01-15T14:32:13|pos_terminal  |2026-04-12 19:27:26.127284|bronze|
|pay_d2e4f6a1|usr_4423def |mer_3321jkl|